# EPJ Data Science: figure analyses and provenance

This notebook executes the manuscript's nine original figure-generation pathways
against the **full-corpus saved analysis outputs**. It does not pretend to retrain
the 5.07-million-post classifier or turn an already viewed test year into an
untouched holdout. Later sections recompute conditional-as-of RQ1 ridge-v2
and model-matched LightGBM sensitivities from saved refit predictions, with
prior-date-only VoteView and thread-closure embargo. These remain conditional
on a retrospectively fixed event/outage cohort, and VoteView release times
are unverified; no operational or causal estimate is claimed. RQ3 member-
bootstrap intervals do not quantify national-transition uncertainty. All
raster figures are regenerated from plotting source at 600 dpi; vector PDFs
are kept.

Run on HPCF with `PIPELINE_ROOT=~/x_box/pipeline` and `EPJDS_WORKSPACE` pointing
to this isolated workspace. The source scripts and output hashes below identify
what each cell actually used. Cells fail on missing inputs rather than fabricate data.

In [ ]:
import os, sys, json, hashlib, runpy, io
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr
import pandas as pd
import matplotlib
from PIL import Image
from IPython.display import display

pipeline = Path(os.environ.get('PIPELINE_ROOT', str(Path.home() / 'x_box/pipeline'))).expanduser().resolve()
workspace = Path(os.environ.get('EPJDS_WORKSPACE', str(pipeline / 'manuscript/epjds_submission_work/notebook'))).expanduser().resolve()
scripts = pipeline / 'scripts/v7'
v7 = pipeline / 'output/v7'
figdir = workspace / 'figures'
logdir = workspace / 'logs'
for folder in (figdir, logdir): folder.mkdir(parents=True, exist_ok=True)
corpus = pipeline / 'data/congresstweets_full/congresstweets_full.parquet'
assert corpus.is_file(), corpus
assert (v7 / 'figs/MANIFEST.md').is_file()
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
print('pipeline', pipeline)
print('corpus bytes', corpus.stat().st_size, 'sha256', sha256(corpus))
print('plotting manifest sha256', sha256(v7 / 'figs/MANIFEST.md'))
print('pandas', pd.__version__, 'matplotlib', matplotlib.__version__)

## Source-to-figure map

The archived plotting script's file stems do not equal every manuscript figure
number: the manuscript calls rhythm Figure 5, although the plotting source
names it `fig6_rhythm_skill`. Captions in the DOCX govern submission numbering.
The figures below replay saved full-corpus analysis tables, not a new training run.

In [ ]:
sources = {
    1: ('fig1_ti7_party_account_type', 'figs_01_party.py'),
    2: ('fig2_spec_curve', 'figs_01_party.py'),
    3: ('fig3_party_trends', 'figs_01_party.py'),
    4: ('fig4_regime_effects', 'figs_02_effects.py'),
    5: ('fig5_forecast_ladder', 'figs_03_forecast.py'),
    6: ('fig6_rhythm_skill', 'figs_03_forecast.py'),
    7: ('fig7_trajectory_skill', 'figs_03_forecast.py'),
    8: ('fig8_lee_case_study', 'figs_04_lee_temporal.py'),
    9: ('fig9_point_process_comparison', 'figs_04_lee_temporal.py'),
}
assert len(sources) == 9
display(pd.DataFrame([{'source_plot_id': i, 'stem': stem, 'source_script': script,
                       'script_sha256': sha256(scripts / script)}
                      for i, (stem, script) in sources.items()]))

## Recreate figures from archived analytical tables

This redirects only the output directory and resolution of the original
read-only plotting scripts. It does not edit the shared scripts or their
input CSV/Parquet files. Source PDF remains vector artwork; PNG is rendered
directly from Matplotlib at 600 dpi and is not an upsample of the old 300-dpi file.

In [ ]:
sys.path.insert(0, str(scripts))
import figs_common as common
common.ROOT = str(pipeline)
common.V7 = str(v7)
common.DATA = str(pipeline / 'data')
common.OUT = str(figdir)
def save600(fig, name):
    fig.savefig(figdir / f'{name}.pdf', bbox_inches='tight')
    fig.savefig(figdir / f'{name}.png', dpi=600, bbox_inches='tight')
    common.plt.close(fig)
    print('saved', name, '600 dpi PNG + vector PDF')
common.save = save600
def run_plot_script(name):
    assert name in {entry[1] for entry in sources.values()}
    buff = io.StringIO()
    with redirect_stdout(buff), redirect_stderr(buff):
        runpy.run_path(str(scripts / name), run_name='__main__')
    (logdir / f'{name}.log').write_text(buff.getvalue(), encoding='utf-8')
    print(name, 'complete; last lines:')
    print('\n'.join(buff.getvalue().splitlines()[-5:]))

In [ ]:
run_plot_script('figs_01_party.py')

In [ ]:
run_plot_script('figs_02_effects.py')

In [ ]:
run_plot_script('figs_03_forecast.py')

In [ ]:
run_plot_script('figs_04_lee_temporal.py')

In [ ]:
figures = sorted(figdir.glob('*.png'))
assert len(figures) == 9, [p.name for p in figures]
records = []
for png in figures:
    pdf = png.with_suffix('.pdf')
    assert pdf.is_file(), pdf
    with Image.open(png) as image:
        dpi = image.info.get('dpi', (0, 0))
        assert min(dpi) >= 599, (png.name, dpi)
        records.append({'stem': png.stem, 'pixels': list(image.size),
                        'dpi': list(dpi), 'png_sha256': sha256(png),
                        'pdf_sha256': sha256(pdf)})
(workspace / 'notebook_figures_manifest.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
display(pd.DataFrame(records)[['stem', 'pixels', 'dpi']])

## Auditable numeric checks on the saved analysis outputs

These cells recompute skill from stored losses. They do not re-fit models.
The annual post-election extension reuses a viewed final year and is
**not an untouched holdout**. Later cells reproduce corrected forecast
sensitivities from saved refits without equating member-level intervals with
uncertainty over independent national transitions.

In [ ]:
p1 = pd.read_csv(v7 / 'p1/skill_test.csv')
r = p1[(p1.task == 'reg') & (p1.model == 'gbm_reg_R') & (p1.baseline == 'gbm_reg_P')].iloc[0]
assert abs((1 - r.score_model / r.score_base) - r.skill) < 1e-8
print('P1 incremental rhythm MSE skill', r.skill, '95% interval', (r.lo, r.hi))
p3 = pd.read_csv(v7 / 'p3b/py/skill_reg_final.csv')
rows = p3[(p3['set'] == 'final') & (p3.target == 'd_ti7') &
          (p3.versus == 'persistence') &
          (((p3.variant == 'i') & (p3.model == 'enet_pw')) |
           ((p3.variant == 'ii') & (p3.model == 'ridge_core_pw')))]
for _, row in rows.iterrows():
    assert abs((1 - row.score_model / row.score_base) - row.skill) < 1e-8
display(rows[['variant', 'model', 'skill', 'lo', 'hi', 'n_rows']])
print('These variants differ in model and information time; their skill difference is not a clean effect estimate.')

In [ ]:
runs = Path(os.environ.get('EPJDS_RUNS_DIR', str(workspace.parent))).expanduser().resolve()
for analysis in ('rq1_strict_v2', 'rq1_gbt_v2', 'rq3', 'external'):
    folder = runs / analysis
    files = sorted(folder.glob('*.json')) if folder.is_dir() else []
    print(analysis, 'result files:', [f.name for f in files] or 'PENDING - no run yet')

## Conditional-as-of RQ1 reconstruction and separate forecast sensitivities

        RQ1 below uses corrected strict-v2 inputs: prior-Eastern-date VoteView
        votes, January 3 noon-Eastern active-Congress transitions, and a
        7,200-second closure embargo for historical labels and prior events.
        The same 13 monthly origin cohorts, 422,529 repaired test rows and
        frozen hyperparameters are used in a model-matched LightGBM rerun.
        The upstream event/outage/thread cohort is retrospectively selected,
        and actual VoteView release times are unverified; these are conditional
        historical sensitivities, not deployment-ready or causal estimates.

        The RQ3 2022 final year was previously examined and is exploratory;
        member-bootstrap intervals do not account for common national-year
        shocks or prospective cohort selection. External 2021 post-runoff
        comparisons include outcome data before the origin and do not make
        future-only annual forecasts.

In [ ]:
runs = Path(os.environ.get('EPJDS_RUNS_DIR', str(workspace.parent))).expanduser().resolve()
newdir = workspace / 'new_figures'
newdir.mkdir(exist_ok=True)
def save_new(fig, stem):
    fig.savefig(newdir / f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(newdir / f'{stem}.png', dpi=600, bbox_inches='tight')
    common.plt.close(fig)

### RQ1: corrected conditional-as-of ridge sensitivity

        Recompute paired error from all 13 strict-v2 prediction files and match
        the stored evaluation. Training labels and prior-event features have a
        7,200-second closure embargo; VoteView uses prior Eastern dates only.
        These fixed retrospective test events are not an online cohort.
        The account-bootstrap interval does not capture common calendar shocks.

In [ ]:
a = runs / 'rq1_strict_v2'
assert sha256(a / 'manifest.json') == '5bfb6cb0289fc840448a3c07dd6a4073488bc05b8792e9c10b6fb4d96ea35a01'
rq1_eval = json.loads((a / 'evaluation.json').read_text())['combined_conditional_strict']['primary']
rq1_paths = sorted((a / 'predictions/combined_conditional_strict').glob('origin_*.parquet'))
assert len(rq1_paths) == 13
rq1_pred = pd.concat([pd.read_parquet(p) for p in rq1_paths], ignore_index=True)
assert rq1_pred.thread_id.is_unique and len(rq1_pred) == rq1_eval['n'] == 422529
mse_content = (rq1_pred.ti7 - rq1_pred.content).pow(2).mean()
mse_rhythm = (rq1_pred.ti7 - rq1_pred.rhythm).pow(2).mean()
assert abs(mse_content - rq1_eval['mse_content']) < 1e-7
assert abs(mse_rhythm - rq1_eval['mse_rhythm']) < 1e-7
assert abs((1 - mse_rhythm / mse_content) - rq1_eval['relative_mse_reduction']) < 1e-6
block = json.loads((a / 'temporal_block_sensitivity.json').read_text())
assert block['n_months'] == 13 and block['positive_gain_months'] == 11
display(pd.DataFrame([{'events': len(rq1_pred), 'MSE content': mse_content,
    'MSE + rhythm': mse_rhythm, 'relative reduction': 1 - mse_rhythm/mse_content,
    'positive months': block['positive_gain_months']}]))
monthly = pd.DataFrame(block['per_month'])
fig, ax = common.plt.subplots(figsize=(7.2, 3.0))
ax.plot(monthly.origin.str[:7], monthly.strict_delta_mse * 1000,
        marker='o', color=common.BLUE)
ax.axhline(0, color='black', linewidth=0.7)
ax.tick_params(axis='x', rotation=50)
ax.set_ylabel('Paired MSE reduction (x 1,000)')
ax.set_title('Conditional-as-of ridge v2; retrospective cohort')
fig.tight_layout()
save_new(fig, 'RQ1_asof_sensitivity')

### RQ1: original-versus-repaired LightGBM on common events

        The archived P1 P/R LightGBM replay matched every original prediction.
        Here the original predictions are restricted to the repaired 422,529
        test events; the new model is refit with corrected v2 features, closure
        eligibility, and frozen P/R hyperparameters. Both contrasts add rhythm
        to a content baseline, but this is a JOINT feature/cohort redesign, not
        an isolated learner or causal effect. Account-bootstrap intervals below
        retain member trajectories but not shared national-time shocks.

In [ ]:
g = runs / 'rq1_gbt_v2'
gbt_eval = json.loads((g / 'aligned_evaluation.json').read_text())
gbt_paths = sorted((g / 'predictions_aligned').glob('origin_*.parquet'))
assert len(gbt_paths) == 13
gbt_pred = pd.concat([pd.read_parquet(p) for p in gbt_paths], ignore_index=True)
assert len(gbt_pred) == gbt_eval['n'] == len(rq1_pred)
assert gbt_pred.thread_id.is_unique
assert gbt_pred.thread_id.sort_values(ignore_index=True).equals(
    rq1_pred.thread_id.sort_values(ignore_index=True))
gbt_rows = []
for prefix, key in [('orig','legacy_matched'), ('aligned','repaired')]:
    p_mse = (gbt_pred.ti7 - gbt_pred[f'{prefix}_p']).pow(2).mean()
    r_mse = (gbt_pred.ti7 - gbt_pred[f'{prefix}_r']).pow(2).mean()
    ref = gbt_eval['models'][key]
    assert abs(p_mse - ref['mse_p']) < 1e-7
    assert abs(r_mse - ref['mse_r']) < 1e-7
    assert abs((1 - r_mse / p_mse) - ref['skill']) < 1e-6
    gbt_rows.append({'model':key, 'n':len(gbt_pred), 'P MSE':p_mse,
        'R MSE':r_mse, 'skill':1-r_mse/p_mse,
        'ci_lo':ref['ci95_skill'][0], 'ci_hi':ref['ci95_skill'][1]})
display(pd.DataFrame(gbt_rows))
vals = pd.DataFrame(gbt_rows)
fig, ax = common.plt.subplots(figsize=(7.2, 3.5))
heights = 100 * vals.skill.to_numpy()
ax.bar([0,1], heights, width=0.48, color=[common.BLUE, common.ORANGE])
ax.errorbar([0,1], heights, yerr=[100*(vals.skill-vals.ci_lo).to_numpy(),
    100*(vals.ci_hi-vals.skill).to_numpy()], fmt='none', ecolor='black', capsize=4)
ax.set_xticks([0,1], ['Original (common rows)', 'Corrected v2 refit'])
ax.set_ylabel('Rhythm MSE skill vs content baseline (%)')
ax.set_title('Matched P1 LightGBM; fixed retrospective cohort')
ax.text(0.5, 0.98, 'Paired change: %.3f pp' %
        (100*gbt_eval['paired_skill_change_repaired_minus_legacy']),
        ha='center', va='top', transform=ax.transAxes, fontsize=9)
fig.tight_layout()
save_new(fig, 'RQ1_gbt_matched_repair')

### RQ3: July full-year and December future-remainder targets

        Each origin compares status-free and status-aware models on matched
        member-years. The reused 2022 year is shown separately. Bootstrap
        intervals resample members, not independent political transitions.

In [ ]:
b = runs / 'rq3'
rq3_pred = pd.read_csv(b / 'predictions.csv')
rq3_skill = pd.read_csv(b / 'skill.csv')
pairs = rq3_skill[(rq3_skill.model == 'ridge_status') & (rq3_skill.baseline == 'ridge_free')].copy()
assert len(pairs) == 4
for row in pairs.itertuples():
    years = [2019, 2020, 2021] if row.set == 'development' else [2022]
    frame = rq3_pred[(rq3_pred.phase == row.phase) & (rq3_pred.py.isin(years))]
    z = {k: (d.y - d.pred).pow(2).mean() ** 0.5 for k, d in frame.groupby('model')}
    assert abs((1 - z['ridge_status']/z['ridge_free']) - row.skill) < 1e-8
    assert len(frame[frame.model == 'ridge_status']) == row.n
display(pairs[['set','phase','n','members','rmse','baseline_rmse','skill','lo','hi']])
fig, ax = common.plt.subplots(figsize=(7.2, 3.4))
for j, (lab, group, color) in enumerate([
    ('Development 2019–21', pairs[pairs['set']=='development'], common.BLUE),
    ('Reused 2022, exploratory', pairs[pairs['set']!='development'], common.ORANGE)]):
    d = group.set_index('phase').loc[['july','update']]
    x = pd.Series([0,1]) + (j-0.5)*0.29
    ax.bar(x, 100*d.skill.to_numpy(), width=0.27, color=color, label=lab)
    ax.errorbar(x, 100*d.skill.to_numpy(),
                yerr=[100*(d.skill-d.lo).to_numpy(),100*(d.hi-d.skill).to_numpy()],
                fmt='none', ecolor='black', capsize=3)
ax.set_xticks([0,1], ['July: full year','Dec 8: future remainder'])
ax.set_ylabel('Status RMSE skill vs status-free (%; higher is better)')
ax.set_title('Retrospective complete-case panel; member-bootstrap intervals')
ax.axhline(0, color='black', linewidth=0.7)
ax.legend(loc='lower right', fontsize=8)
fig.tight_layout()
save_new(fig, 'RQ3_two_origins')

### External panel: 2021 Senate status timing

        This is a same-annual-target sensitivity, not a clean Jan-20-origin
        future-only forecast. The persistence denominator is zero annual change.

In [ ]:
c = runs / 'external'
external = json.loads((c / 'results.json').read_text())
timing = []
for measure in ('tw', 'twfb'):
    pred = pd.read_csv(c / f'predictions_{measure}.csv')
    d = pred[pred.Y == 2021]
    baseline_rmse = d.target.pow(2).mean() ** 0.5
    for when, scenario in [('pre','2020-12-31'),('post','2021-01-20')]:
        rmse = (d.target - d[f'{when}_R_both']).pow(2).mean() ** 0.5
        val = external[measure]['metrics']['scenarios'][scenario]['by_year']['2021']['R_both']['skill_vs_persist']
        assert abs((1 - rmse/baseline_rmse) - val) < 1e-8
        timing.append({'measure': measure, 'clock': scenario, 'members': len(d), 'skill': val})
display(pd.DataFrame(timing))
fig, ax = common.plt.subplots(figsize=(6.4, 3.0))
for j, measure in enumerate(('tw','twfb')):
    vals = [v['skill'] for v in timing if v['measure']==measure]
    ax.bar([i+(j-0.5)*0.3 for i in [0,1]], [100*v for v in vals], width=0.28,
           color=common.BLUE if j==0 else common.ORANGE, label=measure)
ax.set_xticks([0,1], ['Before runoff result','After runoff result'])
ax.set_ylabel('2021 annual RMSE skill vs no-change (%)')
ax.set_title('Conditional timing sensitivity, not a future-only target')
ax.legend()
fig.tight_layout()
save_new(fig, 'External_2021_timing')

### Diagnostic-figure output manifest

        Both raster and vector exports are hashed; raster metadata and dimensions
        are read back from the actual saved PNG rather than inferred from `savefig`.

In [ ]:
from PIL import Image
diagnostic_manifest = []
for f in sorted(newdir.glob('*.png')):
    pdf = f.with_suffix('.pdf')
    assert pdf.is_file()
    with Image.open(f) as im:
        pixels = list(im.size)
        dpi = list(im.info['dpi'])
    assert min(dpi) >= 599 and min(pixels) > 1000
    diagnostic_manifest.append({'stem':f.stem, 'png':f.name, 'pdf':pdf.name,
        'pixels':pixels, 'stored_dpi':dpi, 'png_sha256':sha256(f), 'pdf_sha256':sha256(pdf)})
assert len(diagnostic_manifest) == 4
(workspace/'new_figures_manifest.json').write_text(json.dumps(diagnostic_manifest, indent=2))
display(pd.DataFrame(diagnostic_manifest)[['stem','pixels','stored_dpi']])

### Standalone tables

        These are cell-faithful DOCX exports, not re-estimation of all RVT
        coefficients or classifier training. Their hashes and shapes are checked.

In [ ]:
table_dir = Path(os.environ.get('EPJDS_TABLE_DIR', str(workspace/'tables'))).expanduser().resolve()
table_manifest = json.loads((workspace/'figures_tables.json').read_text())
table_names = ['Table_1.csv','Table_2.csv','Table_3.csv','Table_4.csv'] + [f'Table_S{i}.csv' for i in range(1,7)]
assert len(table_manifest['tables']) == len(table_names) == 10
table_rows = []
for t in table_manifest['tables']:
    name = Path(t['csv'].replace('\\', '/')).name
    assert name in table_names
    f = table_dir/name
    assert f.is_file() and sha256(f) == t['csv_sha256']
    frame = pd.read_csv(f, dtype=str, keep_default_na=False)
    assert len(frame)+1 == t['rows'] and len(frame.columns) == t['columns']
    table_rows.append({'id':t['id'], 'rows':t['rows'], 'columns':t['columns'], 'sha256':t['csv_sha256']})
assert (table_dir/'Table_1.csv').exists()
display(pd.DataFrame(table_rows))

## Upstream reproducibility boundary

`output/v7/figs/MANIFEST.md` records each figure's source analysis table and
original plot script. The plotting sections above have been executed against
those saved outputs; RQ1 refits in isolated run folders are separately
reproduced from their saved predictions. Full reclassification of the
5.07-million-post corpus and non-figure analyses are upstream steps, not
silently implied by a notebook that plots archived CSVs. The conditional
fixed-cohort, VoteView-release and national-year limitations, source scripts,
tests and output hashes must be read together with manuscript claims.